In [45]:
# BLOCK A: 全域設定 (CONFIG) - 所有模組開關都在這裡控制
# ============================================================
import os
import re
import ast
import random
import copy
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import timm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report, confusion_matrix

CONFIG = {
    # ---- 路徑設定 ----
    "META_CSV": "/kaggle/input/datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/ptbxl_database.csv",
    "SCP_CSV": "/kaggle/input/datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/scp_statements.csv",
    "IMG_ROOT": "/kaggle/input/datasets/bjoernjostein/ptb-xl-ecg-image-gmc2024",
    "IMG_RATE": "lr",          # 'lr' (100Hz) 或 'hr' (500Hz)，需與影像資料集實際命名一致
    "MAX_IMG_PER_RECORD": 1,   # 每筆 ecg_id 取幾張合成影像 (1 = 只取 -0，避免資料洩漏疑慮)
    "OUTPUT_DIR": "/kaggle/working/outputs",

    # ---- 任務設定 ----
    "SINGLE_LABEL_ONLY": True,     # True: 只保留單一 superclass 的紀錄 (簡化成單標籤分類)
    "AGE_CLIP_MAX": 90,            # PTB-XL 對 >89 歲設為 300，需 clip

    # ---- 多模態開關 ----
    "USE_AGE_MODALITY": True,      # 關掉即退化為純影像分類 (方便做 ablation 比較)
    "AGE_EMBED_DIM": 16,

    # ---- 模型設定 ----
    "BACKBONE": "resnet18",
    "PRETRAINED": True,
    "NUM_CLASSES": 5,              # NORM / MI / STTC / CD / HYP
    "IMG_SIZE": 224,

    # ---- 訓練設定 ----
    "BATCH_SIZE": 32,
    "NUM_WORKERS": 2,
    "EPOCHS": 30,
    "FREEZE_EPOCHS": 5,            # 前 N epoch 凍結 image backbone，只訓練 fusion head
    "LR_HEAD": 1e-3,
    "LR_BACKBONE": 1e-4,
    "WEIGHT_DECAY": 1e-4,
    "SEED": 42,

    # ---- 不平衡處理開關 ----
    "USE_WEIGHTED_SAMPLER": True,
    "USE_CLASS_WEIGHTED_LOSS": True,
    "USE_FOCAL_LOSS": False,       # 與 USE_CLASS_WEIGHTED_LOSS 二選一，Focal 優先
    "FOCAL_GAMMA": 2.0,

    # ---- 資料增強開關 ----
    "USE_MIXUP": False,            # 對 ECG 印刷影像做 Mixup 需謹慎，預設關閉
    "MIXUP_ALPHA": 0.2,

    # ---- Grad-CAM 開關 ----
    "USE_GRADCAM": True,
    "GRADCAM_TARGET_LAYER": "layer4",  # 依 backbone 調整 (resnet 系列用 layer4)

    # ---- Early stopping ----
    "EARLY_STOP_METRIC": "val_f1_macro",  # 用 F1 而非 accuracy/val_loss 做早停判準
    "EARLY_STOP_PATIENCE": 8,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

print('ok')

ok


In [46]:
# BLOCK B: 隨機種子固定 (可重現性)
# ============================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["SEED"])

print('ok')

ok


In [47]:
# BLOCK C: PTB-XL Metadata 讀取與 diagnostic superclass 聚合
# ============================================================
def load_ptbxl_metadata(meta_csv: str, scp_csv: str, single_label_only: bool = True) -> pd.DataFrame:
    meta = pd.read_csv(meta_csv, index_col="ecg_id")
    meta["scp_codes"] = meta["scp_codes"].apply(ast.literal_eval)

    agg_df = pd.read_csv(scp_csv, index_col=0)
    agg_df = agg_df[agg_df.diagnostic == 1]

    def aggregate_diagnostic(scp_codes: dict):
        classes = set()
        for code in scp_codes.keys():
            if code in agg_df.index:
                classes.add(agg_df.loc[code].diagnostic_class)
        return list(classes)

    meta["diagnostic_superclass"] = meta["scp_codes"].apply(aggregate_diagnostic)
    meta = meta[meta["diagnostic_superclass"].apply(len) > 0].copy()

    if single_label_only:
        meta = meta[meta["diagnostic_superclass"].apply(len) == 1].copy()
        meta["label"] = meta["diagnostic_superclass"].apply(lambda x: x[0])
    else:
        # 多標籤模式：保留 list，訓練時需搭配 BCEWithLogitsLoss (本檔案預設走單標籤路徑)
        meta["label"] = meta["diagnostic_superclass"]

    return meta

print('ok')

ok


In [48]:
# BLOCK D: 年齡清理 + 影像路徑展開 (含資料夾分層規則)
# ============================================================
def get_ptbxl_folder(ecg_id: int) -> str:
    """ecg_id -> 千位分組資料夾名稱，例如 12345 -> '12000'"""
    return f"{(ecg_id // 1000) * 1000:05d}"


def clean_age(meta: pd.DataFrame, clip_max: int) -> pd.DataFrame:
    meta = meta.copy()

    n_before = len(meta)
    meta = meta[meta["age"].notna()].copy()
    n_dropped = n_before - len(meta)
    if n_dropped > 0:
        print(f"[clean_age] 移除 {n_dropped} 筆缺少年齡的紀錄 ({n_before} -> {len(meta)})")

    # PTB-XL 對 >89 歲的紀錄設為 300 歲 (HIPAA 去識別化)，需特別處理
    meta["is_elderly_capped"] = meta["age"] >= 200
    meta["age_clean"] = meta["age"].clip(upper=clip_max)
    return meta


def build_image_index(meta_df: pd.DataFrame, img_root: str, rate: str,
                       max_img_per_record: int) -> pd.DataFrame:
    """展開成 (img_path, ecg_id, patient_id, age_clean, label) 的長表"""
    rows = []
    for ecg_id, row in meta_df.iterrows():
        folder = get_ptbxl_folder(ecg_id)
        for idx in range(max_img_per_record):
            fname = f"{ecg_id:05d}_{rate}-{idx}.png"
            fpath = os.path.join(img_root, folder, fname)
            if os.path.exists(fpath):
                rows.append({
                    "img_path": fpath,
                    "ecg_id": ecg_id,
                    "patient_id": row["patient_id"],
                    "age_clean": row["age_clean"],
                    "label": row["label"],
                })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(
            "沒有找到任何符合命名規則的影像檔案，請先確認 IMG_ROOT / IMG_RATE / 檔名格式是否正確。"
        )
    return df

print('ok')

ok


In [49]:
# BLOCK E: Patient-level train/val/test 切分 (避免資料洩漏)
# ============================================================
def patient_level_split(df: pd.DataFrame, seed: int, val_size=0.15, test_size=0.15):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(df, groups=df["patient_id"]))
    trainval_df = df.iloc[trainval_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    relative_val_size = val_size / (1 - test_size)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=relative_val_size, random_state=seed)
    train_idx, val_idx = next(gss2.split(trainval_df, groups=trainval_df["patient_id"]))
    train_df = trainval_df.iloc[train_idx].reset_index(drop=True)
    val_df = trainval_df.iloc[val_idx].reset_index(drop=True)

    # 檢查 patient_id 不重疊
    assert set(train_df.patient_id) & set(val_df.patient_id) == set()
    assert set(train_df.patient_id) & set(test_df.patient_id) == set()
    assert set(val_df.patient_id) & set(test_df.patient_id) == set()

    return train_df, val_df, test_df


print('ok')

ok


In [50]:
# BLOCK F: 年齡標準化 (用 train set 統計量，避免資訊洩漏)
# ============================================================
class AgeScaler:
    def __init__(self):
        self.mean = None
        self.std = None

    def fit(self, ages: np.ndarray):
        self.mean = float(np.mean(ages))
        self.std = float(np.std(ages) + 1e-6)
        return self

    def transform(self, ages: np.ndarray):
        return (ages - self.mean) / self.std

print('ok')

ok


In [51]:
# BLOCK G: 影像資料增強 (Augmentation)
# ============================================================
def get_transforms(img_size: int):
    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomRotation(3),          # ECG印刷影像角度增強要保守，避免破壞波形判讀
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_tf, eval_tf

print('ok')


ok


In [52]:
# BLOCK H: 多模態 Dataset
# ============================================================
class PTBXLMultimodalDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform, age_scaler: AgeScaler,
                 label2idx: dict, use_age: bool = True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.age_scaler = age_scaler
        self.label2idx = label2idx
        self.use_age = use_age

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["img_path"]).convert("RGB")
        image = self.transform(image)

        if self.use_age:
            age_norm = self.age_scaler.transform(np.array([row["age_clean"]]))[0]
            age_tensor = torch.tensor([age_norm], dtype=torch.float32)
        else:
            age_tensor = torch.tensor([0.0], dtype=torch.float32)  # placeholder，模型會忽略

        label = self.label2idx[row["label"]]
        return image, age_tensor, label


print('ok')



ok


In [53]:
# BLOCK I: WeightedRandomSampler 建構 (處理類別不平衡)
# ============================================================
def build_weighted_sampler(labels: list) -> WeightedRandomSampler:
    class_counts = Counter(labels)
    num_samples = len(labels)
    class_weights = {c: num_samples / count for c, count in class_counts.items()}
    sample_weights = [class_weights[l] for l in labels]
    return WeightedRandomSampler(
        weights=sample_weights, num_samples=num_samples, replacement=True
    )

print('ok')

ok


In [54]:
# BLOCK J: DataLoader 建構
# ============================================================
def build_dataloaders(train_df, val_df, test_df, label2idx, cfg: dict):
    train_tf, eval_tf = get_transforms(cfg["IMG_SIZE"])

    age_scaler = AgeScaler().fit(train_df["age_clean"].values)

    train_ds = PTBXLMultimodalDataset(train_df, train_tf, age_scaler, label2idx, cfg["USE_AGE_MODALITY"])
    val_ds = PTBXLMultimodalDataset(val_df, eval_tf, age_scaler, label2idx, cfg["USE_AGE_MODALITY"])
    test_ds = PTBXLMultimodalDataset(test_df, eval_tf, age_scaler, label2idx, cfg["USE_AGE_MODALITY"])

    if cfg["USE_WEIGHTED_SAMPLER"]:
        train_labels_idx = [label2idx[l] for l in train_df["label"]]
        sampler = build_weighted_sampler(train_labels_idx)
        train_loader = DataLoader(train_ds, batch_size=cfg["BATCH_SIZE"], sampler=sampler,
                                   num_workers=cfg["NUM_WORKERS"])
    else:
        train_loader = DataLoader(train_ds, batch_size=cfg["BATCH_SIZE"], shuffle=True,
                                   num_workers=cfg["NUM_WORKERS"])

    val_loader = DataLoader(val_ds, batch_size=cfg["BATCH_SIZE"], shuffle=False,
                             num_workers=cfg["NUM_WORKERS"])
    test_loader = DataLoader(test_ds, batch_size=cfg["BATCH_SIZE"], shuffle=False,
                              num_workers=cfg["NUM_WORKERS"])

    return train_loader, val_loader, test_loader, age_scaler


print('ok')

ok


In [55]:
# BLOCK K: 多模態模型 (Image Encoder + Age Encoder + Fusion Head)
# ============================================================
class MultimodalECGNet(nn.Module):
    def __init__(self, backbone: str, pretrained: bool, num_classes: int,
                 age_embed_dim: int = 16, use_age: bool = True):
        super().__init__()
        self.use_age = use_age

        self.img_encoder = timm.create_model(backbone, pretrained=pretrained, num_classes=0)
        img_feat_dim = self.img_encoder.num_features

        if use_age:
            self.age_encoder = nn.Sequential(
                nn.Linear(1, age_embed_dim),
                nn.ReLU(),
                nn.Linear(age_embed_dim, age_embed_dim),
                nn.ReLU(),
            )
            fusion_in_dim = img_feat_dim + age_embed_dim
        else:
            self.age_encoder = None
            fusion_in_dim = img_feat_dim

        self.classifier = nn.Sequential(
            nn.Linear(fusion_in_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, image, age):
        img_feat = self.img_encoder(image)
        if self.use_age:
            age_feat = self.age_encoder(age)
            fused = torch.cat([img_feat, age_feat], dim=1)
        else:
            fused = img_feat
        return self.classifier(fused)

    def freeze_backbone(self):
        for p in self.img_encoder.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.img_encoder.parameters():
            p.requires_grad = True


print('ok')

ok


In [56]:
# BLOCK L: Loss function (class-weighted CrossEntropy 或 Focal Loss)
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


def build_criterion(train_labels_idx: list, num_classes: int, cfg: dict):
    counts = Counter(train_labels_idx)
    weights = torch.tensor(
        [len(train_labels_idx) / counts.get(c, 1) for c in range(num_classes)],
        dtype=torch.float32,
    ).to(DEVICE)
    # 常見做法：weight clipping，避免極端不平衡讓多數類別 recall 崩掉
    weights = torch.clamp(weights, min=0.5, max=5.0)

    if cfg["USE_FOCAL_LOSS"]:
        return FocalLoss(alpha=weights, gamma=cfg["FOCAL_GAMMA"])
    elif cfg["USE_CLASS_WEIGHTED_LOSS"]:
        return nn.CrossEntropyLoss(weight=weights)
    else:
        return nn.CrossEntropyLoss()

print('ok')

ok


In [57]:
# BLOCK M: Optimizer + Freeze/Unfreeze 排程
# ============================================================
def build_optimizer(model: MultimodalECGNet, cfg: dict):
    backbone_params = list(model.img_encoder.parameters())
    other_params = list(model.classifier.parameters())
    if model.use_age:
        other_params += list(model.age_encoder.parameters())

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": cfg["LR_BACKBONE"]},
        {"params": other_params, "lr": cfg["LR_HEAD"]},
    ], weight_decay=cfg["WEIGHT_DECAY"])
    return optimizer
print('ok')

ok


In [58]:
# BLOCK N: LR Scheduler
# ============================================================
def build_scheduler(optimizer, cfg: dict):
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["EPOCHS"])

print('ok')

ok


In [59]:
# BLOCK O: Mixup (只作用於影像分支，年齡分支維持原值傳遞，預設關閉)
# ============================================================
def mixup_data(images, ages, labels, alpha: float):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = images.size(0)
    index = torch.randperm(batch_size).to(images.device)

    mixed_images = lam * images + (1 - lam) * images[index]
    # 年齡不做混合，保留原始年齡對應原始影像的邏輯較合理；若要混合可改成加權平均
    labels_a, labels_b = labels, labels[index]
    return mixed_images, ages, labels_a, labels_b, lam


def mixup_criterion(criterion, pred, labels_a, labels_b, lam):
    return lam * criterion(pred, labels_a) + (1 - lam) * criterion(pred, labels_b)


print('ok')

ok


In [60]:
# BLOCK P: 訓練 / 驗證迴圈
# ============================================================
def train_one_epoch(model, loader, optimizer, criterion, cfg: dict):
    model.train()
    total_loss = 0.0
    for batch_idx, (images, ages, labels) in enumerate(loader):
        images, ages, labels = images.to(DEVICE), ages.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        if cfg["USE_MIXUP"]:
            mixed_images, ages, labels_a, labels_b, lam = mixup_data(images, ages, labels, cfg["MIXUP_ALPHA"])
            outputs = model(mixed_images, ages)
            loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
        else:
            outputs = model(images, ages)
            loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)

        if batch_idx % 50 == 0:
            print(f"  batch {batch_idx}/{len(loader)} loss={loss.item():.4f}")

    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, idx2label: dict):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for images, ages, labels in loader:
        images, ages, labels = images.to(DEVICE), ages.to(DEVICE), labels.to(DEVICE)
        outputs = model(images, ages)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    val_loss = total_loss / len(loader.dataset)
    f1_macro = f1_score(all_labels, all_preds, average="macro")

    from sklearn.metrics import precision_score, recall_score, accuracy_score
    precision_macro = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall_macro = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    accuracy = accuracy_score(all_labels, all_preds)

    report = classification_report(
        all_labels, all_preds,
        target_names=[idx2label[i] for i in range(len(idx2label))],
        zero_division=0,
    )
    cm = confusion_matrix(all_labels, all_preds)

    return {
        "val_loss": val_loss,
        "val_f1_macro": f1_macro,
        "val_precision_macro": precision_macro,
        "val_recall_macro": recall_macro,
        "val_accuracy": accuracy,
        "report": report, "cm": cm,
        "all_labels": all_labels, "all_preds": all_preds,
    }

print('ok')

ok


In [61]:
# BLOCK Q: Grad-CAM (只作用於影像分支)
# ============================================================
class GradCAM:
    def __init__(self, model: MultimodalECGNet, target_layer_name: str):
        self.model = model
        self.gradients = None
        self.activations = None

        target_layer = dict([*model.img_encoder.named_modules()])[target_layer_name]
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, image_tensor, age_tensor, class_idx=None):
        self.model.eval()
        image_tensor = image_tensor.unsqueeze(0).to(DEVICE)
        age_tensor = age_tensor.unsqueeze(0).to(DEVICE)

        output = self.model(image_tensor, age_tensor)
        if class_idx is None:
            class_idx = output.argmax(dim=1).item()

        self.model.zero_grad()
        output[0, class_idx].backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=image_tensor.shape[2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

    # 注意: Grad-CAM 只反映影像分支的空間注意力，年齡分支的貢獻無法用此方式視覺化，
    # 解讀結果時務必註明「此為影像模態的注意力，不代表年齡模態的影響程度」。
print('ok')

ok


In [62]:
# BLOCK R: 主流程 (串接所有模組)
# ============================================================
def main(cfg: dict):
    print(f"Device: {DEVICE}")
 
    # --- 資料準備 ---
    meta = load_ptbxl_metadata(cfg["META_CSV"], cfg["SCP_CSV"], cfg["SINGLE_LABEL_ONLY"])
    meta = clean_age(meta, cfg["AGE_CLIP_MAX"])
    long_df = build_image_index(meta, cfg["IMG_ROOT"], cfg["IMG_RATE"], cfg["MAX_IMG_PER_RECORD"])
    print(f"總影像數: {len(long_df)}, 類別分布:\n{long_df['label'].value_counts()}")
 
    labels_sorted = sorted(long_df["label"].unique())
    label2idx = {l: i for i, l in enumerate(labels_sorted)}
    idx2label = {i: l for l, i in label2idx.items()}
 
    train_df, val_df, test_df = patient_level_split(long_df, cfg["SEED"])
    print(f"train/val/test = {len(train_df)}/{len(val_df)}/{len(test_df)}")
 
    train_loader, val_loader, test_loader, age_scaler = build_dataloaders(
        train_df, val_df, test_df, label2idx, cfg
    )
 
    # --- 模型 / loss / optimizer ---
    model = MultimodalECGNet(
        backbone=cfg["BACKBONE"], pretrained=cfg["PRETRAINED"],
        num_classes=cfg["NUM_CLASSES"], age_embed_dim=cfg["AGE_EMBED_DIM"],
        use_age=cfg["USE_AGE_MODALITY"],
    ).to(DEVICE)
 
    train_labels_idx = [label2idx[l] for l in train_df["label"]]
    criterion = build_criterion(train_labels_idx, cfg["NUM_CLASSES"], cfg)
    optimizer = build_optimizer(model, cfg)
    scheduler = build_scheduler(optimizer, cfg)
 
    # --- 訓練迴圈 (含 freeze/unfreeze + early stopping) ---
    best_metric = -np.inf
    best_state = None
    best_epoch = None
    patience_counter = 0
    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_f1_macro": [],
               "val_precision_macro": [], "val_recall_macro": [], "val_accuracy": []}
 
    for epoch in range(cfg["EPOCHS"]):
        if epoch < cfg["FREEZE_EPOCHS"]:
            model.freeze_backbone()
        else:
            model.unfreeze_backbone()
 
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, cfg)
        val_metrics = evaluate(model, val_loader, criterion, idx2label)
        scheduler.step()
 
        print(f"[Epoch {epoch+1}/{cfg['EPOCHS']}] "
              f"train_loss={train_loss:.4f} val_loss={val_metrics['val_loss']:.4f} "
              f"val_f1_macro={val_metrics['val_f1_macro']:.4f} "
              f"val_precision={val_metrics['val_precision_macro']:.4f} "
              f"val_recall={val_metrics['val_recall_macro']:.4f}")
 
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["val_loss"])
        history["val_f1_macro"].append(val_metrics["val_f1_macro"])
        history["val_precision_macro"].append(val_metrics["val_precision_macro"])
        history["val_recall_macro"].append(val_metrics["val_recall_macro"])
        history["val_accuracy"].append(val_metrics["val_accuracy"])
 
        current_metric = val_metrics[cfg["EARLY_STOP_METRIC"]]
        if current_metric > best_metric:
            best_metric = current_metric
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= cfg["EARLY_STOP_PATIENCE"]:
                print(f"Early stopping at epoch {epoch+1} (best {cfg['EARLY_STOP_METRIC']}={best_metric:.4f})")
                break
 
    # --- 載回最佳權重，跑 test set ---
    model.load_state_dict(best_state)
    test_metrics = evaluate(model, test_loader, criterion, idx2label)
    print("\n=== Test Set 結果 ===")
    print(test_metrics["report"])
    print("Confusion Matrix:\n", test_metrics["cm"])
 
    torch.save(model.state_dict(), os.path.join(cfg["OUTPUT_DIR"], "best_model.pth"))
 
    # --- 自動產出所有圖表 ---
    class_names = [idx2label[i] for i in range(len(idx2label))]
    generate_all_plots(
        history=history,
        freeze_epochs=cfg["FREEZE_EPOCHS"],
        best_epoch=best_epoch,
        test_metrics=test_metrics,
        all_labels=test_metrics["all_labels"],
        all_preds=test_metrics["all_preds"],
        class_names=class_names,
        output_dir=cfg["OUTPUT_DIR"],
    )
 
    # --- Grad-CAM 範例 (只示範第一筆 test 影像) ---
    if cfg["USE_GRADCAM"]:
        sample_image, sample_age, sample_label = test_loader.dataset[0]
        gradcam = GradCAM(model, cfg["GRADCAM_TARGET_LAYER"])
        cam, pred_idx = gradcam.generate(sample_image, sample_age)
        print(f"Grad-CAM 範例完成，預測類別: {idx2label[pred_idx]}, 真實類別: {idx2label[sample_label]}")
        np.save(os.path.join(cfg["OUTPUT_DIR"], "sample_gradcam.npy"), cam)
 
    return model, test_metrics
print('ok')

ok


In [63]:
# BLOCK S: 視覺化 (訓練曲線 / 混淆矩陣 / 各類別指標)
# ============================================================
def setup_cjk_font():
    """設定中文字型，避免 matplotlib CJK 缺字警告。找不到就靜默回退英文。"""
    import matplotlib
    from matplotlib import font_manager
    candidates = [
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc",
    ]
    for path in candidates:
        if os.path.exists(path):
            font_manager.fontManager.addfont(path)
            name = font_manager.FontProperties(fname=path).get_name()
            matplotlib.rcParams["font.family"] = name
            matplotlib.rcParams["axes.unicode_minus"] = False
            return True
    return False
 
 
def plot_training_curves(history: dict, freeze_epochs: int, best_epoch: int, output_dir: str):
    import matplotlib.pyplot as plt
    epochs = history["epoch"]
 
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
    axes[0].plot(epochs, history["train_loss"], marker="o", label="Train Loss", color="#E2B08C")
    axes[0].plot(epochs, history["val_loss"], marker="s", label="Val Loss", color="#8C9EE2")
    axes[0].axvline(x=freeze_epochs, color="gray", linestyle="--", alpha=0.6,
                     label=f"Backbone unfreeze (epoch {freeze_epochs})")
    axes[0].axvline(x=best_epoch, color="green", linestyle=":", alpha=0.8,
                     label=f"Best checkpoint (epoch {best_epoch})")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training / Validation Loss")
    axes[0].legend(fontsize=9)
    axes[0].grid(alpha=0.3)
 
    axes[1].plot(epochs, history["val_f1_macro"], marker="D", color="#C97B63")
    axes[1].axvline(x=freeze_epochs, color="gray", linestyle="--", alpha=0.6)
    axes[1].axvline(x=best_epoch, color="green", linestyle=":", alpha=0.8)
    best_idx = epochs.index(best_epoch)
    axes[1].scatter([best_epoch], [history["val_f1_macro"][best_idx]], color="green", s=100, zorder=5)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Val F1 (macro)")
    axes[1].set_title("Validation F1-macro")
    axes[1].grid(alpha=0.3)
 
    plt.tight_layout()
    path = os.path.join(output_dir, "01_training_curves.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
 
def plot_confusion_matrix(cm: np.ndarray, class_names: list, output_dir: str):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
 
    im0 = axes[0].imshow(cm, cmap="Oranges")
    axes[0].set_xticks(range(len(class_names)))
    axes[0].set_yticks(range(len(class_names)))
    axes[0].set_xticklabels(class_names)
    axes[0].set_yticklabels(class_names)
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")
    axes[0].set_title("Confusion Matrix (count)")
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = cm[i, j]
            color = "white" if val > cm.max() * 0.5 else "black"
            axes[0].text(j, i, str(val), ha="center", va="center", color=color, fontsize=10)
    plt.colorbar(im0, ax=axes[0], fraction=0.046)
 
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    im1 = axes[1].imshow(cm_norm, cmap="Oranges", vmin=0, vmax=1)
    axes[1].set_xticks(range(len(class_names)))
    axes[1].set_yticks(range(len(class_names)))
    axes[1].set_xticklabels(class_names)
    axes[1].set_yticklabels(class_names)
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")
    axes[1].set_title("Confusion Matrix (row-normalized)")
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = cm_norm[i, j]
            color = "white" if val > 0.5 else "black"
            axes[1].text(j, i, f"{val:.0%}", ha="center", va="center", color=color, fontsize=10)
    plt.colorbar(im1, ax=axes[1], fraction=0.046)
 
    plt.tight_layout()
    path = os.path.join(output_dir, "02_confusion_matrix.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
 
def plot_per_class_metrics(all_labels, all_preds, class_names: list, output_dir: str):
    import matplotlib.pyplot as plt
    from sklearn.metrics import precision_recall_fscore_support
 
    precision, recall, f1, support = precision_recall_fscore_support(
        all_labels, all_preds, labels=range(len(class_names)), zero_division=0
    )
 
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(class_names))
    width = 0.25
 
    bars1 = ax.bar(x - width, precision, width, label="Precision", color="#E2B08C")
    bars2 = ax.bar(x, recall, width, label="Recall", color="#C97B63")
    bars3 = ax.bar(x + width, f1, width, label="F1-score", color="#8C9EE2")
 
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            h = bar.get_height()
            ax.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                        xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)
 
    ax.set_xlabel("Class")
    ax.set_ylabel("Score")
    ax.set_title("Per-class Precision / Recall / F1 (Test Set)")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{c}\n(n={s})" for c, s in zip(class_names, support)])
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
 
    plt.tight_layout()
    path = os.path.join(output_dir, "03_per_class_metrics.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
    return precision, recall, f1, support
 
 
def plot_imbalance_vs_recall(support, recall, class_names: list, output_dir: str):
    import matplotlib.pyplot as plt
    fig, ax1 = plt.subplots(figsize=(10, 6))
 
    ax1.bar(class_names, support, color="#D9CFC1", alpha=0.7, label="Test set support")
    ax1.set_xlabel("Class")
    ax1.set_ylabel("Support (# samples)", color="#8C7B6C")
    ax1.tick_params(axis="y", labelcolor="#8C7B6C")
 
    ax2 = ax1.twinx()
    ax2.plot(class_names, recall, marker="o", color="#C0392B", linewidth=2.5, markersize=8)
    ax2.set_ylabel("Recall", color="#C0392B")
    ax2.tick_params(axis="y", labelcolor="#C0392B")
    ax2.set_ylim(0, 1.0)
    for i, r in enumerate(recall):
        ax2.annotate(f"{r:.2f}", (i, r), textcoords="offset points", xytext=(0, 10),
                     ha="center", color="#C0392B", fontweight="bold")
 
    ax1.set_title("Class Imbalance vs Recall")
    fig.tight_layout()
    path = os.path.join(output_dir, "04_imbalance_vs_recall.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
 
def plot_yolo_style_results(history: dict, freeze_epochs: int, best_epoch: int, output_dir: str):
    """仿 YOLO results.png 風格：一張圖網格顯示所有指標隨 epoch 的折線走勢"""
    import matplotlib.pyplot as plt
    epochs = history["epoch"]
    best_idx = epochs.index(best_epoch)
 
    panels = [
        ("train_loss", "Train Loss", "#E2B08C"),
        ("val_loss", "Val Loss", "#8C9EE2"),
        ("val_precision_macro", "Val Precision (macro)", "#C97B63"),
        ("val_recall_macro", "Val Recall (macro)", "#6BA383"),
        ("val_f1_macro", "Val F1 (macro)", "#C0392B"),
        ("val_accuracy", "Val Accuracy", "#8E6C88"),
    ]
 
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    axes = axes.flatten()
 
    for ax, (key, title, color) in zip(axes, panels):
        values = history[key]
        ax.plot(epochs, values, marker="o", markersize=4, color=color, linewidth=1.8)
        # 用平滑線 (簡單移動平均) 疊加，YOLO 風格會有一條淡色 raw + 一條平滑趨勢線
        if len(values) >= 5:
            window = 3
            smoothed = np.convolve(values, np.ones(window) / window, mode="valid")
            smooth_epochs = epochs[window - 1:]
            ax.plot(smooth_epochs, smoothed, color=color, linewidth=2.5, alpha=0.9, linestyle="--")
        ax.axvline(x=freeze_epochs, color="gray", linestyle=":", alpha=0.5)
        ax.axvline(x=best_epoch, color="green", linestyle=":", alpha=0.7)
        ax.scatter([best_epoch], [values[best_idx]], color="green", s=60, zorder=5)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Epoch", fontsize=9)
        ax.grid(alpha=0.3)
 
    fig.suptitle(
        f"Training Results  (backbone unfreeze @ epoch {freeze_epochs}, best checkpoint @ epoch {best_epoch})",
        fontsize=13, fontweight="bold"
    )
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    path = os.path.join(output_dir, "00_results_grid.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"[plot] 儲存: {path}")
 
 
def generate_all_plots(history: dict, freeze_epochs: int, best_epoch: int,
                        test_metrics: dict, all_labels, all_preds,
                        class_names: list, output_dir: str):
    """訓練/評估完成後一次呼叫，產出全部圖表並存到 output_dir"""
    setup_cjk_font()
    os.makedirs(output_dir, exist_ok=True)
 
    plot_yolo_style_results(history, freeze_epochs, best_epoch, output_dir)
    plot_training_curves(history, freeze_epochs, best_epoch, output_dir)
    plot_confusion_matrix(test_metrics["cm"], class_names, output_dir)
    precision, recall, f1, support = plot_per_class_metrics(all_labels, all_preds, class_names, output_dir)
    plot_imbalance_vs_recall(support, recall, class_names, output_dir)
    print(f"\n所有圖表已儲存至: {output_dir}")

print('ok')

ok


In [64]:
model, test_metrics = main(CONFIG)

Device: cuda
[clean_age] 移除 38 筆缺少年齡的紀錄 (16272 -> 16234)
總影像數: 16105, 類別分布:
label
NORM    9012
MI      2509
STTC    2376
CD      1676
HYP      532
Name: count, dtype: int64
train/val/test = 11299/2394/2412


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

  batch 0/354 loss=1.6260
  batch 50/354 loss=1.5756
  batch 100/354 loss=1.4952
  batch 150/354 loss=1.4934
  batch 200/354 loss=1.5024
  batch 250/354 loss=1.4844
  batch 300/354 loss=1.5042
  batch 350/354 loss=1.4824
[Epoch 1/30] train_loss=1.4876 val_loss=1.6389 val_f1_macro=0.2154 val_precision=0.1619 val_recall=0.3880
  batch 0/354 loss=1.4507
  batch 50/354 loss=1.3499
  batch 100/354 loss=1.3173
  batch 150/354 loss=1.5515
  batch 200/354 loss=1.3468
  batch 250/354 loss=1.3496
  batch 300/354 loss=1.2257
  batch 350/354 loss=1.2544
[Epoch 2/30] train_loss=1.4030 val_loss=1.7177 val_f1_macro=0.1977 val_precision=0.1559 val_recall=0.3858
  batch 0/354 loss=1.3812
  batch 50/354 loss=1.3701
  batch 100/354 loss=1.4707
  batch 150/354 loss=1.3479
  batch 200/354 loss=1.5492
  batch 250/354 loss=1.5085
  batch 300/354 loss=1.4915
  batch 350/354 loss=1.2943
[Epoch 3/30] train_loss=1.3688 val_loss=1.6074 val_f1_macro=0.2206 val_precision=0.1603 val_recall=0.3936
  batch 0/354 loss=

KeyboardInterrupt: 